In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- RBF features ----------
class RBF2D:
    def __init__(self, n_per_dim=6, sigma=0.18):
        xs = np.linspace(0, 1, n_per_dim)
        ys = np.linspace(0, 1, n_per_dim)
        self.centers = np.array([(x,y) for x in xs for y in ys], dtype=np.float32)
        self.sigma = float(sigma)
        self.M = len(self.centers)

    def phi(self, x, y):
        xy = np.array([x,y], dtype=np.float32)
        d2 = np.sum((self.centers - xy[None,:])**2, axis=1)
        return np.exp(-0.5 * d2 / (self.sigma**2)).astype(np.float32)

# ---------- map grid cell -> continuous (x,y) ----------
def cell_center_xy(r, c, H=4, W=4):
    x = (c + 0.5) / W
    y = (r + 0.5) / H
    return x, y

# ---------- build representations for all 16 cells ----------
H = W = 4
rbf = RBF2D(n_per_dim=6, sigma=0.18)

cell_xy = []
Phi = []  # shape (16, M)
for r in range(H):
    for c in range(W):
        x,y = cell_center_xy(r,c,H,W)
        cell_xy.append((x,y))
        Phi.append(rbf.phi(x,y))
Phi = np.stack(Phi, axis=0)  # (16, M)

# normalize reps for cosine similarity / PCA
Phi_norm = Phi / (np.linalg.norm(Phi, axis=1, keepdims=True) + 1e-12)

# ---------- (1) Similarity matrix: "how similar do two cells look?" ----------
Sim = Phi_norm @ Phi_norm.T  # cosine similarity (16x16)

plt.figure(figsize=(5,4))
plt.imshow(Sim, origin='lower', vmin=0, vmax=1)
plt.colorbar(label="cosine similarity")
plt.title("Representation similarity between cells")
plt.xlabel("cell index (r*4+c)")
plt.ylabel("cell index (r*4+c)")
plt.tight_layout()
plt.show()

# ---------- (2) 2D embedding via PCA: "geometry of representation space" ----------
# simple PCA with SVD (no sklearn needed)
X = Phi_norm - Phi_norm.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
Z = X @ Vt[:2].T  # (16,2)

plt.figure(figsize=(5,5))
Zx, Zy = Z[:,0], Z[:,1]
plt.scatter(Zx, Zy)

for idx, (r,c) in enumerate([(i//4, i%4) for i in range(16)]):
    plt.text(Zx[idx]+0.01, Zy[idx]+0.01, f"({r},{c})", fontsize=8)

plt.title("PCA of cell representations φ(x,y)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.axis("equal")
plt.tight_layout()
plt.show()

# ---------- (3) "Which RBF is most active" per cell (argmax feature index) ----------
# This shows a discrete "codebook" view of the representation
argmax_feat = np.argmax(Phi, axis=1).reshape(H, W)

plt.figure(figsize=(4.5,4))
plt.imshow(argmax_feat, origin='lower')
plt.colorbar(label="argmax feature index")
plt.title("Most active RBF feature per cell")
plt.xlabel("col")
plt.ylabel("row")
plt.tight_layout()
plt.show()

# ---------- (optional) visualize one feature's receptive field over the plane ----------
for feat_i in range(rbf.centers.shape[0]):
    print(f"Feature {feat_i} center: {rbf.centers[feat_i]}")
    # feat_i = 10  # choose any 0..M-1
    grid_n = 200
    xs = np.linspace(0, 1, grid_n)
    ys = np.linspace(0, 1, grid_n)
    XX, YY = np.meshgrid(xs, ys)

    # compute feature i activation over plane
    ci = rbf.centers[feat_i]
    Z = np.exp(-0.5*((XX-ci[0])**2 + (YY-ci[1])**2)/(rbf.sigma**2))

    plt.figure(figsize=(5,4))
    plt.imshow(Z, origin='lower', extent=[0,1,0,1])
    plt.scatter([ci[0]],[ci[1]], marker='x')
    plt.title(f"One RBF feature receptive field (i={feat_i}, center={tuple(np.round(ci,2))})")
    plt.xlabel("x"); plt.ylabel("y")
    plt.colorbar(label="activation")
    plt.tight_layout()
    plt.show()
